# Lab 11 · Scaling Study · pick a variant, sweep, plot, write up

Every parallel implementation in this course produces numbers. Lab 11 is where you turn those numbers into a **scaling study**: pick one variant, run it at 1, 2, 4, 8, 16, ..., N units (threads, ranks, or GPUs), produce strong-scaling and weak-scaling plots, and write a short report of what you found and why.

**Prerequisites.** One working parallel variant from labs 03, 06, 07, 08, or 10.

**Builds toward.** Lab 12 (final project — extend the code in one direction of your choice).

> **📚 Where to look when you're stuck**
>
> - [**Amdahl vs Gustafson**](https://en.wikipedia.org/wiki/Amdahl%27s_law) — the two views of "speedup"
> - [**Weak vs strong scaling**](https://hpc-wiki.info/hpc/Scaling) — HPC-Wiki explanation
> - [**Efficient use of Crux for a scaling study**](https://docs.alcf.anl.gov/crux/queueing-and-running-jobs/running-jobs/)



## How this notebook works

Same three surfaces as prior labs: **[Hub]**, **[Hub -> cluster]**, **[cluster compute]**.


In [ ]:
# [Hub] Shared toolkit.
from labHelpers import *


### Set up this lab's identity


In [ ]:
# [Hub] Change HPC_USER; re-run.
env = setupLab(labName="lab11", host="crux",
               remoteUser=os.environ.get("HPC_USER","CHANGE_ME"),
               project="UIC-CS455-Sp2027", queue="debug",
               scratch=f"/eagle/UIC-CS455-Sp2027/{os.environ.get('HPC_USER','CHANGE_ME')}")
labDir = pathlib.Path(env['labDir'])


### Preflight


In [ ]:
# [Hub] Reachability + prerequisite artifact.
preflight([
    check("passwordless ssh", sshReachable()),
    check("scheduler answers", schedulerAnswers()),
    check("lab11 dir on cluster", remoteFileExists(env['HPC_LAB_DIR']),
          hint="next cell creates it if missing"),
], infoRows=[('cluster', clusterHost()), ('you', env.get('HPC_USER','?')),
             ('project', env.get('HPC_PROJECT','?')),
             ('lab dir', env.get('HPC_LAB_DIR','?'))])


In [ ]:
# [Hub -> cluster] Make the lab dir if missing.
sshRun(f'mkdir -p {env["HPC_LAB_DIR"]}/out', quiet=True)
print('lab11 dir ready')


## Part 1 · Pick your variant

Which lab's binary will you scale?

| Variant | Scale axis | Recommended range |
|---|---|---|
| OpenMP (lab 03) | threads on one node | 1, 2, 4, ..., 128 |
| MPI (lab 06) | ranks across nodes | 8, 16, 32, ..., 256 |
| Hybrid (lab 07) | nodes at optimal split | 1, 2, 4, 8, 16 |
| Multi-GPU (lab 10) | GPUs at 4/node | 1, 4, 16, 64 |

Pick one. The rest of the lab assumes your choice.


In [ ]:
# [Hub] Set the variant this run uses.
VARIANT = 'hybrid'   # or 'openmp', 'mpi', 'gpu-omp', 'mpi-cuda'
print(f'this scaling study uses variant = {VARIANT}')


In [ ]:
checkpoint("Part 1 - variant chosen", [
    check("variant set", lambda: (VARIANT is not None, VARIANT)),
])


## Part 2 · Strong scaling · fixed problem size

Fix N = 4096 (or largest that fits on your target). Vary the number of workers. Every run appends one row to `timings.csv`.


In [ ]:
# [Hub -> cluster] Sweep. Template - fill in the mpiexec/OMP call for your variant.
counts = [1, 2, 4, 8, 16, 32] if VARIANT == 'openmp' else [8, 16, 32, 64, 128]
jobBody = f'''cd {env["HPC_LAB_DIR"]}
cp ../lab0*/heat2D* . 2>/dev/null || true
N=4096
for w in {" ".join(str(c) for c in counts)}; do
  case "$VARIANT" in
    openmp) OMP_NUM_THREADS=$w OMP_PROC_BIND=close OMP_PLACES=cores \\
            ./heat2Domp --N $N --steps 200 --snapEvery 0 --outDir ./out --variant openmp ;;
    mpi)    mpiexec -n $w --ppn 8 ./heat2Dmpi --N $N --steps 200 ;;
    hybrid) mpiexec -n $(( w * 2 )) --ppn 2 --depth 64 --cpu-bind depth \\
            ./heat2Dhyb --N $N --steps 200 ;;
  esac
done
'''
pbsPath = labDir/'strongJob.pbs'
pbsPath.write_text(pbsHeader(name='lab11Strong', project=env['HPC_PROJECT'],
                             queue=env.get('HPC_QUEUE','debug'),
                             select='16:system=crux', walltime='01:00:00',
                             filesystems='home:eagle',
                             outPath=env['HPC_LAB_DIR']+'/strong.out') + jobBody)
sshPut(str(pbsPath), env['HPC_LAB_DIR']+'/strongJob.pbs')
# jobID = submitJob(env['HPC_LAB_DIR']+'/strongJob.pbs')
print('script staged; uncomment submitJob() line when ready (this is a long run)')


In [ ]:
checkpoint("Part 2 - strong scaling script", [
    check("strong PBS script exists locally", fileExists(str(labDir/'strongJob.pbs'))),
])


## Part 3 · Weak scaling · fixed work per worker

Each worker gets the same amount of work (say 512x512 tile), and N grows with the number of workers. Ideal weak-scaling efficiency is flat 1.0.


In [ ]:
# [Hub -> cluster] Weak sweep - N scales with sqrt(workers).
print('For weak scaling, set N = 512 * int(sqrt(workers)) at each step.')


In [ ]:
checkpoint("Part 3 - weak scaling recipe", [
    check("lab dir", dirExists(str(labDir))),
])


## Part 4 · Produce the two figures

labDD Part 4 + `plotScaling(kind='strong')` and `plotScaling(kind='weak')`.


In [ ]:
# [Hub] Assuming you have timings.csv from Parts 2+3.
if (labDir/'timings.csv').exists():
    r1 = plotScaling(str(labDir/'timings.csv'), kind='strong', variantFilter=VARIANT,
                     baselineCol='threads' if VARIANT=='openmp' else 'ranks',
                     timeCol='wall_s', outPath=str(labDir/'figures'/'strong'))
    print('strong:', [str(p) for p in r1])
else:
    print('run Part 2 first to produce timings.csv')


In [ ]:
checkpoint("Part 4 - figures produced", [
    check("lab dir", dirExists(str(labDir))),
])


## Part 5 · Interpret · what did the numbers say?

Write two paragraphs for your report, one per figure. Address:

1. **Where does speedup deviate from ideal?** Name the number of workers at which measured speedup drops below 80% of ideal.
2. **Why does it deviate?** Use the roofline (lab 02), `perf stat` numbers (also lab 02), and the OpenMP pitfalls list (lab 04) to give a *plausible mechanism*. Not "communication overhead" as a black box — say which operation, on which axis, and how it scales.


In [ ]:
checkpoint("Part 5 - interpretation drafted", [
    check("lab dir persistent", dirExists(str(labDir))),
])


## Part 6 · Bridge to lab 12

Lab 12 is your **final project**: pick one direction to extend the code. Options include a different physics (add a second scalar field, add advection so it becomes advection-diffusion), a different algorithm (implicit solve, AMR), or a different platform (port to a machine you don't have easy access to and describe the port). Lab 13 is the final writeup.


## Wrap up

Moved the spine forward one lab.


### Lab scorecard


In [ ]:
labSummary("Scaling Study")


---
### One-minute feedback

What worked, what didn't, what should be clearer.


In [ ]:
feedback("Scaling Study")
